In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(81426, 31)


In [2]:
retailer_features = (
    df.groupby("customerId")
    .agg(
        total_orders=(
            "orderNumber",
            "nunique"
        ),

        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_products=(
            "skuNumber",
            "nunique"
        ),

        last_order=(
            "createdAt",
            "max"
        ),

        first_order=(
            "createdAt",
            "min"
        ),

        shop_type=(
            "shopType",
            "first"
        ),

        retailer_type=(
            "retailerType",
            "first"
        ),

        hub_name=(
            "hubName",
            "first"
        )
    )
    .reset_index()
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name
0,USR-100,5,16,9,2026-01-21,2026-01-08,Paan A,HVLF,Crossline Events (Noida)
1,USR-1000,2,4,4,2026-01-24,2026-01-10,General B,HVLF,Instant Foods(Noida)
2,USR-100185,2,2,1,2026-01-16,2026-01-15,Paan B,HVHF,Instant Foods(Noida)
3,USR-1002,13,32,12,2026-01-31,2026-01-07,General C,HVHF,Crossline Events (Noida)
4,USR-100211,8,18,12,2026-01-31,2026-01-16,Chemist A,HVHF,Instant Foods (SED)


In [3]:
TODAY = df["createdAt"].max()

retailer_features[
    "days_since_last_order"
] = (
    TODAY -
    retailer_features["last_order"]
).dt.days

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order
0,USR-100,5,16,9,2026-01-21,2026-01-08,Paan A,HVLF,Crossline Events (Noida),10
1,USR-1000,2,4,4,2026-01-24,2026-01-10,General B,HVLF,Instant Foods(Noida),7
2,USR-100185,2,2,1,2026-01-16,2026-01-15,Paan B,HVHF,Instant Foods(Noida),15
3,USR-1002,13,32,12,2026-01-31,2026-01-07,General C,HVHF,Crossline Events (Noida),0
4,USR-100211,8,18,12,2026-01-31,2026-01-16,Chemist A,HVHF,Instant Foods (SED),0


In [4]:
fav_category = (
    df.groupby("customerId")
    ["category"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_category.columns = [
    "customerId",
    "favorite_category"
]

fav_category.head()

,customerId,favorite_category
0,USR-100,Zarda
1,USR-1000,Confectionery
2,USR-100185,Zarda
3,USR-1002,Mouth Fresheners
4,USR-100211,Namkeen


In [5]:
retailer_features = (
    retailer_features.merge(
        fav_category,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category
0,USR-100,5,16,9,2026-01-21,2026-01-08,Paan A,HVLF,Crossline Events (Noida),10,Zarda
1,USR-1000,2,4,4,2026-01-24,2026-01-10,General B,HVLF,Instant Foods(Noida),7,Confectionery
2,USR-100185,2,2,1,2026-01-16,2026-01-15,Paan B,HVHF,Instant Foods(Noida),15,Zarda
3,USR-1002,13,32,12,2026-01-31,2026-01-07,General C,HVHF,Crossline Events (Noida),0,Mouth Fresheners
4,USR-100211,8,18,12,2026-01-31,2026-01-16,Chemist A,HVHF,Instant Foods (SED),0,Namkeen


In [6]:
fav_brand = (
    df.groupby("customerId")
    ["brand"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_brand.columns = [
    "customerId",
    "favorite_brand"
]

In [7]:
retailer_features = (
    retailer_features.merge(
        fav_brand,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category,favorite_brand
0,USR-100,5,16,9,2026-01-21,2026-01-08,Paan A,HVLF,Crossline Events (Noida),10,Zarda,Tulsi
1,USR-1000,2,4,4,2026-01-24,2026-01-10,General B,HVLF,Instant Foods(Noida),7,Confectionery,DS Group
2,USR-100185,2,2,1,2026-01-16,2026-01-15,Paan B,HVHF,Instant Foods(Noida),15,Zarda,Tulsi
3,USR-1002,13,32,12,2026-01-31,2026-01-07,General C,HVHF,Crossline Events (Noida),0,Mouth Fresheners,DS Group
4,USR-100211,8,18,12,2026-01-31,2026-01-16,Chemist A,HVHF,Instant Foods (SED),0,Namkeen,Haldiram


In [8]:
retailer_features[
    "spend_segment"
] = pd.qcut(
    retailer_features[
        "total_qty"
    ],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

retailer_features[
    [
        "customerId",
        "total_qty",
        "spend_segment"
    ]
].head()

,customerId,total_qty,spend_segment
0,USR-100,16,Medium
1,USR-1000,4,Low
2,USR-100185,2,Low
3,USR-1002,32,High
4,USR-100211,18,Medium


In [9]:
retailer_features.to_parquet(
    "../data/processed/retailer_features.parquet",
    index=False
)

print("Saved")

Saved


In [10]:
retailer_features[
    "spend_segment"
].value_counts()

spend_segment
Low       1986
High      1785
Medium    1650
Name: count, dtype: int64

In [11]:
retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category,favorite_brand,spend_segment
0,USR-100,5,16,9,2026-01-21,2026-01-08,Paan A,HVLF,Crossline Events (Noida),10,Zarda,Tulsi,Medium
1,USR-1000,2,4,4,2026-01-24,2026-01-10,General B,HVLF,Instant Foods(Noida),7,Confectionery,DS Group,Low
2,USR-100185,2,2,1,2026-01-16,2026-01-15,Paan B,HVHF,Instant Foods(Noida),15,Zarda,Tulsi,Low
3,USR-1002,13,32,12,2026-01-31,2026-01-07,General C,HVHF,Crossline Events (Noida),0,Mouth Fresheners,DS Group,High
4,USR-100211,8,18,12,2026-01-31,2026-01-16,Chemist A,HVHF,Instant Foods (SED),0,Namkeen,Haldiram,Medium


In [12]:
retailer_features["spend_segment"].value_counts()

spend_segment
Low       1986
High      1785
Medium    1650
Name: count, dtype: int64

In [13]:
retailer_features.shape

(5421, 13)

In [14]:
import pandas as pd

retailer_features = pd.read_parquet(
    "../data/processed/retailer_features.parquet"
)

print(retailer_features.shape)

(5421, 13)


In [15]:
from pathlib import Path

print(Path("../data/processed/retailer_features.parquet").resolve())

C:\Users\Rishit\Desktop\O2R-recommender\data\processed\retailer_features.parquet


In [16]:
print(retailer_features.shape)
print(retailer_features["customerId"].nunique())

(5421, 13)
5421
